# Decision Tree Classifier
## Real-world scenario: Will an employee leave the company?

An HR team wants to predict **employee attrition** - will an employee **stay (0)** or **leave (1)** - based on satisfaction, salary, workload and years at the company. Decision Trees are popular here because the rules they learn are easy to explain to managers.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
60 employees. `department` is a **text** column, so later we will encode it to numbers.

In [ ]:
n = 60
satisfaction = np.random.uniform(0.1, 1.0, n).round(2)   # 0=unhappy, 1=happy
monthly_salary = np.random.randint(3000, 12000, n)
projects       = np.random.randint(1, 8, n)              # workload
years_at_company = np.random.randint(1, 12, n)
department = np.random.choice(['Sales', 'Tech', 'HR'], n)

# Low satisfaction + heavy workload -> more likely to leave
leave_score = (1 - satisfaction) + (projects / 8) - (monthly_salary / 12000) \
              + np.random.normal(0, 0.2, n)
left = (leave_score > leave_score.mean()).astype(int)    # 1 = left

df = pd.DataFrame({
    'satisfaction': satisfaction, 'monthly_salary': monthly_salary,
    'projects': projects, 'years_at_company': years_at_company,
    'department': department, 'left': left
})

# Messy data on purpose
df.loc[4, 'satisfaction'] = np.nan
df.loc[8, 'department'] = np.nan
df = pd.concat([df, df.iloc[[2]]], ignore_index=True)
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nLeft vs stayed:\n', df['left'].value_counts())

### Step 4 - Clean the data
Numeric gaps -> median. Text gaps -> most common value (mode).

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['satisfaction'] = df['satisfaction'].fillna(df['satisfaction'].median())
df['department']   = df['department'].fillna(df['department'].mode()[0])
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Encode the text column
Models need numbers, so we convert `department` into 0/1 columns (one-hot encoding).

In [ ]:
df_encoded = pd.get_dummies(df, columns=['department'], drop_first=True)
df_encoded.head()

### Step 6 - Features (X) and target (y)

In [ ]:
X = df_encoded.drop('left', axis=1)
y = df_encoded['left']

### Step 7 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

### Step 8 - Train the Decision Tree
`max_depth=3` keeps the tree shallow so it stays readable and does not overfit such a small dataset.

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

### Step 9 - Evaluate

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, zero_division=0))

### Step 10 - Visualise the tree
The best part of a Decision Tree: we can read the exact rules it learned.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(model, feature_names=X.columns, class_names=['Stay', 'Leave'],
          filled=True, rounded=True)
plt.title('Employee attrition decision tree'); plt.show()